In [1]:
import sys
import importlib

project_path = r"D:\DFUC_Project"

if project_path not in sys.path:
    sys.path.append(project_path)

%load_ext autoreload
%autoreload 2

In [2]:
import torch

print(torch.__version__)
print("Torch working")

2.12.1+cpu
Torch working


In [3]:
import os
print(os.path.exists(r"D:\DFUC_Project"))

True


In [4]:
import os
print(os.path.exists(r"D:\DFUC_Project\src"))
print(os.path.exists(r"D:\DFUC_Project\src\models\custom_cnn.py"))

True
True


In [5]:
import subprocess
result = subprocess.run(['where', '/r', 'D:\\', 'custom_cnn.py'], capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

D:\DFUC_Project\src\models\custom_cnn.py




In [6]:
import os
print(os.path.exists(r"D:\DFUC_Project\src\models\custom_cnn.py"))

True


In [7]:
import sys
import torch

project_path = r"D:\DFUC_Project"

if project_path not in sys.path:
    sys.path.append(project_path)

from src.models.custom_cnn import CustomCNNDetector

In [8]:
import sys
import torch

project_path = r"D:\DFUC_Project"

if project_path not in sys.path:
    sys.path.append(project_path)

from src.models.custom_cnn import CustomCNNDetector

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CustomCNNDetector().to(device)

print("Device:", device)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU found, using CPU")
print(model)

Device: cpu
GPU: No GPU found, using CPU
CustomCNNDetector(
  (features): Sequential(
    (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(16, 32, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (dense1): Linear(in_features=65536, out_features=16, bias=True)
  (relu): ReLU()
  (dense2): Linear(in_features=16, out_features=5, bias=True)
)


In [9]:
dummy_image = torch.randn(1, 3, 256, 256).to(device)

with torch.no_grad():
    output = model(dummy_image)

print("Input shape:", dummy_image.shape)
print("Output shape:", output.shape)
print("Device:", output.device)

Input shape: torch.Size([1, 3, 256, 256])
Output shape: torch.Size([1, 5])
Device: cpu


In [10]:
from src.dataset import load_annotations

train_image_path = r"D:\DFUC_Project\Dataset\DFUC2020\DFUC_Training_Set\images"
annotation_path = r"D:\DFUC_Project\Dataset\DFUC2020\DFUC_Training_Set\groundtruth.csv"

annotations = load_annotations(annotation_path)

print("Total annotations:", len(annotations))
print("Training images:", annotations["filename"].nunique())

Total annotations: 2496
Training images: 2000


In [11]:
print("Sample annotations:" , annotations.head())

Sample annotations:    filename  xmin  ymin  xmax  ymax
0    100001   270   186   303   239
1    100001    37   176   123   256
2    100002    57   203   178   310
3    100003   267   151   317   203
4    100004   271   163   327   205


In [12]:
from torch.utils.data import DataLoader
from src.dataset import DFUCDataset

dataset = DFUCDataset(
    train_image_path,
    annotations
)

dataloader = DataLoader(
    dataset,
    batch_size=8,
    shuffle=True
)

images, targets = next(iter(dataloader))

images = images.to(device)
targets = targets.to(device)

with torch.no_grad():
    predictions = model(images)

print("Images:", images.shape)
print("Targets:", targets.shape)
print("Predictions:", predictions.shape)
print("Prediction device:", predictions.device)

Images: torch.Size([8, 3, 256, 256])
Targets: torch.Size([8, 5])
Predictions: torch.Size([8, 5])
Prediction device: cpu


In [13]:
from src.losses import DetectionLoss

criterion = DetectionLoss().to(device)

total_loss, object_loss, box_loss = criterion(
    predictions,
    targets
)

print("Total loss:", total_loss.item())
print("Object loss:", object_loss.item())
print("Box loss:", box_loss.item())

Total loss: 1.7464518547058105
Object loss: 0.793381929397583
Box loss: 0.1906139999628067


In [14]:
print("Prediction shape:", predictions.shape)  # must print torch.Size([8, 5])

Prediction shape: torch.Size([8, 5])
